In [39]:
import time
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document
from langchain_chroma import Chroma
from datetime import datetime


c:\Users\Priyanshi Garg\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\priyanshi-garg\AG0853_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
pdf_folder_location = "tesla-annual-reports"

In [47]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [48]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [44]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [49]:
len(tesla_10k_chunks)

3337

In [50]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

### Database Creation

In [51]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [52]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [53]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Priyanshi Garg\AppData\Local\Temp\ipykernel_1820\4093584008.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [54]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [55]:
chromadb_client.heartbeat()

1780488876563048400

In [56]:
chromadb_client.count_collections()

1

In [57]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [58]:
chromadb_client.count_collections()

1

In [59]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [ ]:
# i = 0 # Initialize the starting index for the chunks

# while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
#     vectorstore.add_documents( # Add documents to the vector store in batches of 500
#         documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
#         ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
#     )

#     i += 500 # Increment the index by 500 to move to the next batch
#     time.sleep(15) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [60]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [61]:
from openai import OpenAI
client = OpenAI(
    api_key=os.environ['GEMINI_API_KEY'],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [62]:
model = "gemini-2.5-flash"

In [63]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [64]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

### Query Expansion

In [72]:
query_expansion_system_message = """
You are a financial domain expert assisting in answering questions related to 10-K reports.

Perform query expansion on the question below.

If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, return 3 versions of the query with different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return ONLY valid JSON in the following format:

{
  "expanded_queries": [
    "query 1",
    "query 2",
    "query 3"
  ]
}

Do not include any explanation, markdown, or extra text.
"""

In [73]:
user_message_template = """
<Question>
{question}
</Question>
"""

In [74]:
user_input = "What was the automotive revenue in 2021?"

In [75]:
prompt = [
    {'role':'system', 'content': query_expansion_system_message},
    {'role': 'user', 'content': user_message_template.format(
        question = user_input
       )
    }
]

In [76]:
prompt

[{'role': 'system',
  'content': '\nYou are a financial domain expert assisting in answering questions related to 10-K reports.\n\nPerform query expansion on the question below.\n\nIf there are multiple common ways of phrasing a user question or common synonyms for key words in the question, return 3 versions of the query with different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn ONLY valid JSON in the following format:\n\n{\n  "expanded_queries": [\n    "query 1",\n    "query 2",\n    "query 3"\n  ]\n}\n\nDo not include any explanation, markdown, or extra text.\n'},
 {'role': 'user',
  'content': '\n<Question>\nWhat was the automotive revenue in 2021?\n</Question>\n'}]

In [ ]:
query_expansions = client.chat.completions.create(model=model,
                                                  messages=prompt,
                                                  temperature=0)

In [77]:
print(query_expansions.choices[0].message.content)

```json
{
  "expanded_queries": [
    "What was the automotive revenue in 2021?",
    "What was the auto revenue in 2021?",
    "What were the automotive sales in 2021?",
    "What were the auto sales in 2021?",
    "What was the automotive revenue for fiscal year 2021?",
    "What were the auto sales for fiscal year 2021?",
    "How much automotive revenue was generated in 2021?",
    "How much auto sales were reported in 2021?"
  ]
}
```


In [78]:
import json
import re

# response from LLM
raw_response = query_expansions.choices[0].message.content
response = raw_response.strip()

# Some models return fenced JSON or extra prose around the payload.
if response.startswith("```"):
    response = re.sub(r"^```(?:json)?\s*|\s*```$", "", response, flags=re.S)

json_text = response
json_match = re.search(r"\{.*\}", response, flags=re.S)
if json_match:
    json_text = json_match.group(0)

# convert string to python dict
expanded_queries = json.loads(json_text)

# save to file
with open("expanded_queries.json", "w", encoding="utf-8") as f:
    json.dump(expanded_queries, f, indent=4, ensure_ascii=False)

print("Saved successfully!")

Saved successfully!


In [79]:
print(query_expansions)

ChatCompletion(id='8w0gaovXBO2L4-EPtZyewQo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n  "expanded_queries": [\n    "What was the automotive revenue in 2021?",\n    "What was the auto revenue in 2021?",\n    "What were the automotive sales in 2021?",\n    "What were the auto sales in 2021?",\n    "What was the automotive revenue for fiscal year 2021?",\n    "What were the auto sales for fiscal year 2021?",\n    "How much automotive revenue was generated in 2021?",\n    "How much auto sales were reported in 2021?"\n  ]\n}\n```', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1780485622, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=145, prompt_tokens=169, total_tokens=613, completion_tokens_details=None, prompt_tokens_details=None))


In [80]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split('\n')


In [81]:
query_expansions_list

['```json',
 '{',
 '  "expanded_queries": [',
 '    "What was the automotive revenue in 2021?",',
 '    "What was the auto revenue in 2021?",',
 '    "What were the automotive sales in 2021?",',
 '    "What were the auto sales in 2021?",',
 '    "What was the automotive revenue for fiscal year 2021?",',
 '    "What were the auto sales for fiscal year 2021?",',
 '    "How much automotive revenue was generated in 2021?",',
 '    "How much auto sales were reported in 2021?"',
 '  ]',
 '}',
 '```']

In [82]:
expanded_context_list = []

In [83]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [84]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [85]:
expanded_context_list

['☐\n\t\t\t\t\nNo\n\t\t\n☒\nIndicate\tby\tcheck\tmark\twhether\tthe\tregistrant\t(1)\thas\tfiled\tall\treports\trequired\tto\tbe\tfiled\tby\tSection\t13\tor\t15(d)\tof\tthe\tSecurities\tExchange\tAct\tof\t1934\t(“Exchange\tAct”)\n\t\nduring\tthe\tpreceding\t12\tmonths\t(or\tfor\tsuch\tshorter\tperiod\tthat\tthe\tregistrant\twas\trequired\tto\tfile\tsuch\treports),\tand\t(2)\thas\tbeen\tsubject\tto\tsuch\tfiling\trequirements\tfor\tthe\tpast\n\t\n90\tdays.\t\t\t\t\nYes\n\t\t\n☒\n\t\t\t\tNo\t\t\n☐\nIndicate\tby\tcheck\tmark\twhether\tthe\tregistrant\thas\tsubmitted\telectronically\tevery\tInteractive\tData\tFile\trequired\tto\tbe\tsubmitted\tpursuant\tto\tRule\t405\tof\tRegulation\tS-T\n\t\n(§232.405\tof\tthis\tchapter)\tduring\tthe\tpreceding\t12\tmonths\t(or\tfor\tsuch\tshorter\tperiod\tthat\tthe\tregistrant\twas\trequired\tto\tsubmit\tsuch\tfiles).\t\t\t\t\nYes\n\t\t\n☒\n\t\t\t\tNo\t\t\n☐\nIndicate\tby\tcheck\tmark\twhether\tthe\tregistrant\tis\ta\tlarge\taccelerated\tfiler,\tan\tacce

In [86]:
final_context_documents = set(expanded_context_list)

In [87]:
len(final_context_documents)

32

In [88]:
final_context_documents

{'(Dollars\tin\tmillions)\n2023\n2022\n2021\n$\n%\n$\n%\nCost\tof\trevenues\nAutomotive\tsales\n$\n65,121\t\n$\n49,599\t\n$\n32,415\t\n$\n15,522\t\n31\t\n%\n$\n17,184\t\n53\t\n%\nAutomotive\tleasing\n1,268\t\n1,509\t\n978\t\n(241)\n(16)\n%\n531\t\n54\t\n%\nTotal\tautomotive\tcost\tof\trevenues\n66,389\t\n51,108\t\n33,393\t\n15,281\t\n30\t\n%\n17,715\t\n53\t\n%\nServices\tand\tother\n7,830\t\n5,880\t\n3,906\t\n1,950\t\n33\t\n%\n1,974\t\n51\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\ncost\tof\trevenues\n74,219\t\n56,988\t\n37,299\t\n17,231\t\n30\t\n%\n19,689\t\n53\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\n4,894\t\n3,621\t\n2,918\t\n1,273\t\n35\t\n%\n703\t\n24\t\n%\nTotal\tcost\tof\trevenues\n$\n79,113\t\n$\n60,609\t\n$\n40,217\t\n$\n18,504\t\n31\t\n%\n$\n20,392\t\n51\t\n%\nGross\tprofit\ttotal\tautomotive\n$\n16,030\t\n$\n20,354\t\n$\n13,839\t\nGross\tmargin\ttotal\tautomotive\n19.4\t\n%\n28.5\t\n%\n29.3\t\n%\nGross\tprofit\ttotal\tautomotive\t&\tservices\tand\tothe

In [89]:
system_message = """
You are a helpful AI assistant.
Answer only from the provided context.
If the answer is not in the context, say so.
"""

user_input = "What was the automotive revenue in 2021?"

context = "\n\n".join(final_context_documents)

prompt = f"""

Context:
{context}

Question:
{user_input}

Answer:
"""

print(prompt)



Context:
[
On	acknowledgement	copy
]
	
To:
[●]
as	Collateral	Agent	[
ADDRESS
]
	
Copy
	
to:
[
NAME	OF	EACH
	
CHARGOR
]
	
We	acknowledge	receipt	of	the	above	notice.	We	confirm	and	agree	to	the	matters	referred	to	in	it.
	
	
	
for	and	on	behalf	of
[
Name	of	Account	Bank
]
	
Dated:	[●]

35

“
Effective	Date	TEO	Subsidiary
”	means	any	direct	or	indirect	subsidiary	of	TEO	as	of	the	Effective	Date.
“
Electronic	Signature
”	means	an	electronic	sound,	symbol,	or	process	attached	to,	or	associated	with,	a	contract
	
or	other	record	and	adopted	by	a	Person	with	the	intent	to	sign,	authenticate	or	accept	such	contract	or	record.
“
EMU
”	means	Economic	and	Monetary	Union	as	contemplated	in	the	Treaty.
“
Energy	Environmental	Attribute
”	means	any	credit,	benefit,	reduction,	offset	or	allowance	(such	as	so-called
	
renewable	energy	certificates,	green	tags,	green	certificates,	and	renewable	energy	credits),	howsoever	entitled	or	named,
	
resulting	from,	attributable	to	or	associated	with	the	stor

In [46]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Automotive revenue in 2021 was $47,232 million.
